# **NLP Practical 4 — N-Gram Language Models**

---

**Name: Shubham Sarwar**  
**PRN: 202301040127**

**Aim:** Build unigram, bigram and trigram language models from a text corpus,
compute probabilities (with and without Laplace smoothing), generate sentences
probabilistically, and evaluate the model using **perplexity**.

**Key concept (N-gram):** An N-gram is a sequence of N consecutive tokens.  
- Unigram (N=1): single words → `('the',)`  
- Bigram  (N=2): pairs of consecutive words → `('the','city')`  
- Trigram (N=3): triples → `('the','city','was')`

An N-gram **language model** estimates  
&nbsp;&nbsp;&nbsp; P(next word | previous N−1 words)  
from how often those sequences appear in the training corpus.


## **1. Imports**

In [ ]:
import nltk
import re
import math
import random

# Download data files needed by NLTK's tokenizer (only first run)
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.util import ngrams                                  # builds n-grams from a list of tokens
from nltk.probability import FreqDist                         # like Counter, but with NLTK utilities
from nltk.lm.preprocessing import padded_everygram_pipeline   # prepares data for nltk.lm models
from nltk.lm import MLE                                       # MLE = Maximum-Likelihood-Estimate LM

## **2. Text Corpus**

In [ ]:
sentences = """
The morning after the upheaval of the night protests, the city was surreally quiet. Waking in the parking garage, Eleanor lifted herself out of the nest of old coats and her backpack on the floor of the backseat. As she drove away from the one sanctuary she could think of as streets were shut down, Eleanor saw evidence of the night's violence in the strewn litter, broken glass, and the watchful police presence.

She had always believed that the truth was a fixed point, but now it felt like a shadow at the edge of every moment. People in this town paid attention to everyone else's details: a car missing from a driveway, a skipped shift at work, or one less body tucked into the pew on Sunday morning. This was how they governed themselves with the understanding that there was no such thing as a secret.

Eleanor turned the radio dial, but only static filled the air. She thought of the library back home, filled with books that smelled of old paper and dust. She wondered if her own story would ever be written, or if it would simply vanish, washed away by an alien sea of indifference.
"""

## **3. Preprocessing (Tokenization + Cleaning)**

Steps:
1. Split corpus into **sentences**.
2. Split each sentence into **words**, lowercase them, keep only alphabetic words
   (drop punctuation, apostrophes etc.).
3. Add special tokens `<s>` and `</s>` at the start and end of every sentence — these
   markers help the model learn what words typically **start** or **end** a sentence.


In [ ]:
# 1) sentence split
individual_sentences = sent_tokenize(sentences)
print(f"Number of sentences: {len(individual_sentences)}\n")

# 2) word tokenise + lowercase + keep only letters
tokenized = [
    [w for w in word_tokenize(s.lower()) if w.isalpha()]
    for s in individual_sentences
]

print("=" * 60)
print("TOKENIZED SENTENCES")
print("=" * 60)
for i, t in enumerate(tokenized, 1):
    print(f"Sentence {i}: {t}")

# 3) add <s> and </s>
tokenized_with_tokens = [['<s>'] + tokens + ['</s>'] for tokens in tokenized]

## **4. Generate N-grams**

We slide a window of size N over the tokens and collect every window as a tuple.
NLTK's `ngrams(tokens, n)` does this for us.


In [ ]:
all_unigrams, all_bigrams, all_trigrams = [], [], []

for tokens in tokenized_with_tokens:
    all_unigrams += list(ngrams(tokens, 1))    # produces tuples like ('the',)
    all_bigrams  += list(ngrams(tokens, 2))    # ('the','morning')
    all_trigrams += list(ngrams(tokens, 3))    # ('the','morning','after')

print("=" * 60)
print("N-GRAM GENERATION (showing first 10 of each)")
print("=" * 60)
print("Unigrams:", all_unigrams[:10])
print("\nBigrams :", all_bigrams[:10])
print("\nTrigrams:", all_trigrams[:10])

## **5. Frequency Counts**

`FreqDist` counts how many times each n-gram occurs. This is the foundation of every
probability we will compute.


In [ ]:
unigram_counts = FreqDist(all_unigrams)
bigram_counts  = FreqDist(all_bigrams)
trigram_counts = FreqDist(all_trigrams)

print("=" * 60)
print("TOP 10 MOST FREQUENT N-GRAMS")
print("=" * 60)
print("\nUnigrams:")
for ng, c in unigram_counts.most_common(10):
    print(f"  {ng} : {c}")

print("\nBigrams:")
for ng, c in bigram_counts.most_common(10):
    print(f"  {ng} : {c}")

print("\nTrigrams:")
for ng, c in trigram_counts.most_common(10):
    print(f"  {ng} : {c}")

## **6. Probability Calculation**

### Unigram probability
$$P(w) = \frac{\text{count}(w)}{\text{total words}}$$

### Bigram probability (Maximum Likelihood Estimate, MLE)
$$P(w_2 \mid w_1) = \frac{\text{count}(w_1, w_2)}{\text{count}(w_1)}$$

### Why **Laplace (add-one) smoothing**?
If a bigram NEVER appears in training, MLE gives probability **0**, which destroys
the whole sentence probability (multiplying by zero). To avoid this we pretend we
saw every bigram one extra time:

$$P_{\text{Laplace}}(w_2 \mid w_1) = \frac{\text{count}(w_1, w_2) + 1}{\text{count}(w_1) + V}$$

where `V` is the size of the vocabulary.


In [ ]:
# Vocabulary = set of distinct words (excluding sentence markers? we keep them).
vocab = set(w[0] for w in unigram_counts)
V = len(vocab)
print(f"Vocabulary size V = {V}")

# ---- Unigram probability ----
# Total words in the corpus (sum of all unigram counts)
TOTAL_WORDS = sum(unigram_counts.values())

def unigram_prob(word):
    """P(word) = count(word) / total words."""
    return unigram_counts.get((word,), 0) / TOTAL_WORDS

# ---- Bigram probability ----
def bigram_prob(w1, w2, smoothing=True):
    """P(w2 | w1).  smoothing=True uses Laplace add-one."""
    count = bigram_counts.get((w1, w2), 0)
    base  = unigram_counts.get((w1,), 0)

    if smoothing:
        return (count + 1) / (base + V)
    else:
        return count / base if base != 0 else 0

# ---- Trigram probability ----
def trigram_prob(w1, w2, w3, smoothing=True):
    """P(w3 | w1, w2)."""
    count = trigram_counts.get((w1, w2, w3), 0)
    base  = bigram_counts.get((w1, w2), 0)

    if smoothing:
        return (count + 1) / (base + V)
    else:
        return count / base if base != 0 else 0

# ---- Demo ----
print("\nUnigram Probabilities:")
for w in ["the", "she", "city", "eleanor"]:
    print(f"  P({w}) = {unigram_prob(w):.4f}")

print("\nBigram Probabilities:")
for w1, w2 in [("the","city"), ("of","the"), ("she","was"), ("the","air")]:
    p_raw    = bigram_prob(w1, w2, smoothing=False)
    p_smooth = bigram_prob(w1, w2, smoothing=True)
    print(f"  P({w2}|{w1})  Raw: {p_raw:.4f}   Laplace: {p_smooth:.4f}")

print("\nTrigram Probabilities (Laplace smoothed):")
for w1, w2, w3 in [("the","city","was"), ("she","was","in"), ("the","air","was")]:
    print(f"  P({w3}|{w1},{w2}) = {trigram_prob(w1, w2, w3, True):.4f}")

## **7. Sentence Generation (Probabilistic)**

To generate a sentence:
1. Start with the sentence-start token `<s>`.
2. Look at all bigrams `(<s>, x)` — these tell us how likely each word `x` is to be
   the FIRST word of a sentence.
3. Randomly pick one of those next words, **weighted by their probability**
   (more probable words are picked more often → realistic-looking text).
4. Repeat using the new word as the context, stop when we hit `</s>` or hit a max length.


In [ ]:
def generate_sentence(max_words=15):
    word = "<s>"
    sentence = []

    for _ in range(max_words):
        # Find every bigram whose FIRST word is the current word.
        # The probability of going from `word` to candidate `w2` is bigram_prob(word, w2).
        candidates = {w2: bigram_prob(word, w2, smoothing=True)
                      for (w1, w2) in bigram_counts if w1 == word}

        if not candidates:           # safety stop if nothing follows
            break

        # random.choices() does weighted random selection
        word = random.choices(
            list(candidates.keys()),
            weights=list(candidates.values())
        )[0]

        if word == "</s>":           # natural end of sentence
            break

        sentence.append(word)

    return " ".join(sentence)


print("=" * 60)
print("GENERATED SENTENCES")
print("=" * 60)
for i in range(5):
    print(f"Sentence {i+1}: {generate_sentence()}")

## **8. NLTK Built-in MLE Model (alternative way)**

NLTK also has a ready-made Maximum-Likelihood-Estimate language model.  
Here we train it as a bigram model and use it to generate words.


In [ ]:
# padded_everygram_pipeline returns (training_data, padded_vocab) for an n-gram model.
# Argument 2 = train a bigram model.
train_data, vocab_nltk = padded_everygram_pipeline(2, tokenized)

model = MLE(2)                      # bigram MLE language model
model.fit(train_data, vocab_nltk)

print("NLTK MLE-generated sequences:")
print(model.generate(7, text_seed=['forest']))
print(model.generate(7, text_seed=['eleanor']))

## **9. Perplexity — How Good is the Model?**

**Perplexity (PP)** measures how *surprised* the model is by a new sentence.  
- **Lower perplexity → better model** (the sentence is well-predicted).  
- **Higher perplexity → worse** (the sentence was unexpected).

Formula for a sentence with $N$ bigrams:
$$PP(W) = 2^{-\frac{1}{N}\sum_{i=1}^{N} \log_2 P(w_i \mid w_{i-1})}$$

We use Laplace-smoothed probabilities so unknown words don't give $\log_2(0) = -\infty$.


In [ ]:
def perplexity(sentence):
    """Compute bigram perplexity of a sentence using Laplace-smoothed probabilities."""
    words = ['<s>'] + sentence.lower().split() + ['</s>']
    log_prob = 0.0
    N = len(words) - 1                                    # number of bigrams

    for i in range(1, len(words)):
        p = bigram_prob(words[i-1], words[i], smoothing=True)
        log_prob += math.log2(p)

    return 2 ** (-log_prob / N)


test_sentences = [
    "the morning was quiet",       # words appear in corpus -> lower perplexity
    "the air was filled",          # mostly in-vocab
    "eleanor drove the car",       # mix of known words
    "quantum blockchain neural"    # OUT-OF-VOCABULARY -> high perplexity
]

print("=" * 60)
print("PERPLEXITY RESULTS  (lower = model is more confident)")
print("=" * 60)
for s in test_sentences:
    print(f"{s:35s} -> Perplexity: {perplexity(s):.2f}")